# Ensemble Rejection Sampling — Reproducing all examples

This notebook provides a self-contained JAX implementation of **Ensemble Rejection Sampling** (ERS) from:

> Deligiannidis, Doucet, Thornton & Rubenthaler, *Ensemble Rejection Sampling*, technical report.

We implement and reproduce results for all three examples in Section 5 of the paper:
1. **Conditioned random walks** (Table 1)
2. **Non-linear time series / Kitagawa model** (time-inhomogeneous $f_t$)
3. **Stochastic volatility model** (S&P 500 data)

Each example plugs in via four arguments:
- `w_init_fn`: `x_0 [N,...] -> (w [N], log_scale)`
- `weight_matrix_fn`: `(x_prev, x_t, t_idx) -> (W [N,N], log_scale)`
- `log_transition_fn`: `(x_next, x_prevs, t_next) -> log_p [N]`
- `wbar`: `[T]` array of upper bounds

For time-homogeneous models, `weight_matrix_fn` simply ignores `t_idx`
and returns `log_scale = 0.0`.


In [ ]:
# @title Imports and setup
import jax
import jax.numpy as jnp
import numpy as np
import functools
import time
import pandas as pd

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## Generic ERS engine

**Algorithm overview** (Algorithm 3 in the paper):
1. Sample $X_t^1, \ldots, X_t^N \overset{\mathrm{iid}}{\sim} q_t$ for $t = 1, \ldots, T$.
2. Run **forward filter** to compute $\widehat{Z}$ and filtering probabilities.
3. **Backward sample** a trajectory $X_{1:T}^{K_{1:T}}$.
4. Run **bound filter**: replace weights at selected indices with upper bounds $\overline{w}_t$ to get $\overline{Z}$.
5. Accept with probability $\widehat{Z}/\overline{Z}$.



In [ ]:
# @title Generic ERS engine (supports time-inhomogeneous models)

def ers_engine(rng, xs, w_init_fn, weight_matrix_fn, log_transition_fn, wbar):
  """Generic ERS engine for time-(in)homogeneous HMMs.

  Args:
    rng: JAX PRNG key.
    xs: [T, N, ...] proposal particles (pre-sampled from q_t).
    w_init_fn: x_0 [N, ...] -> (w [N], log_scale scalar).
    weight_matrix_fn: (x_prev [N,...], x_t [N,...], t_idx int) -> (W [N,N], log_scale scalar).
    log_transition_fn: (x_next [...], x_prevs [N,...], t_next int) -> log_p [N].
    wbar: [T] array, upper bounds wbar_t for each time step.

  Returns:
    accept: bool, x_traj: [T, ...], K: [T] selected indices
  """
  T, N = xs.shape[:2]
  rng, rng_back, rng_ar = jax.random.split(rng, 3)

  def forward_filter(bound_indices=None):
    w0, ls0 = w_init_fn(xs[0])
    if bound_indices is not None:
      wbar0_s = jnp.exp(jnp.log(wbar[0]) - ls0)
      w0 = w0.at[bound_indices[0]].set(wbar0_s)
    il0 = jnp.sum(w0)
    fp0 = w0 / il0
    log_lik = ls0 + jnp.log(il0)

    def fwd_body(carry, inputs):
      log_lik, fp_prev, x_prev = carry
      t_idx, x_t = inputs
      W, ls = weight_matrix_fn(x_prev, x_t, t_idx)
      if bound_indices is not None:
        bi = bound_indices[t_idx]
        bpi = bound_indices[t_idx - 1]
        wbar_s = jnp.exp(jnp.log(wbar[t_idx]) - ls)
        W = W.at[bpi, :].set(wbar_s)
        W = W.at[:, bi].set(wbar_s)
        W = W.at[bpi, bi].set(wbar_s)
      jp = fp_prev @ W
      il = jnp.sum(jp)
      return (log_lik + ls + jnp.log(il), jp / il, x_t), jp / il

    (log_lik_final, _, _), fps_rest = jax.lax.scan(
        fwd_body, (log_lik, fp0, xs[0]),
        (jnp.arange(1, T), xs[1:]))
    fps = jnp.concatenate([fp0[None, :], fps_rest], axis=0)
    return log_lik_final, fps

  # 1. Forward filter (no bounds)
  log_lik_final, fps = forward_filter()

  # 2. Backward sampling
  rng_bT, rng_brest = jax.random.split(rng_back)
  K_T = jax.random.choice(rng_bT, N, (), p=fps[-1])

  def bwd_body(carry, inputs):
    x_next, _ = carry
    t_idx, rng_t = inputs
    log_tp = log_transition_fn(x_next, xs[t_idx], t_idx + 1)
    lp = jnp.log(fps[t_idx] + 1e-30) + log_tp
    lp = lp - jax.scipy.special.logsumexp(lp)
    K_t = jax.random.choice(rng_t, N, (), p=jnp.exp(lp))
    return (xs[t_idx, K_t], K_t), K_t

  brngs = jax.random.split(rng_brest, T - 1)
  _, K_rest = jax.lax.scan(
      bwd_body, (xs[T-1, K_T], K_T),
      (jnp.arange(T-2, -1, -1), brngs))
  K = jnp.concatenate([K_rest[::-1], jnp.array([K_T])])

  # 3. Forward filter with bounds
  log_lik_b_final, _ = forward_filter(bound_indices=K)

  # 4. Accept/reject
  accept = jnp.log(jax.random.uniform(rng_ar)) < (log_lik_final - log_lik_b_final)
  x_traj = xs[jnp.arange(T), K]
  return accept, x_traj, K


def run_experiment(ers_step_fn, rng, num_trials=1000, batch_n=1):
  """Run multiple ERS trials and return acceptance rate, std, and timing."""
  @functools.partial(jax.jit, static_argnums=(1,))
  def sample_n(rng, n):
    rngs = jax.random.split(rng, n)
    return jax.vmap(lambda r: ers_step_fn(r)[0])(rngs)

  _ = sample_n(rng, batch_n).block_until_ready()  # warmup

  all_accepts = []
  tic = time.time()
  for _ in range(-(-num_trials // batch_n)):
    rng, rng_s = jax.random.split(rng)
    acc = sample_n(rng_s, batch_n)
    all_accepts.append(acc)
  jax.block_until_ready(all_accepts)
  toc = time.time()

  all_accepts = jnp.concatenate(all_accepts)
  p_acc = jnp.mean(all_accepts)
  std_acc = jnp.std(all_accepts) / jnp.sqrt(len(all_accepts))
  return float(p_acc), float(std_acc), toc - tic


print('Generic ERS engine defined.')


## Example 1: Conditioned random walks (Section 5.1)

**Model** (particle in absorbing medium on $\mathscr{S} = [0,1]^d$):
- $\mu(x) = \mathscr{U}(x; \mathscr{S})$
- $f(x' \mid x) = \mathscr{N}(x'; x, \sigma_v^2 I_d)$
- $G_t(x) = \mathbf{1}_{\mathscr{S}}(x)$

**Proposals**: $q_t(x) = \mathscr{U}(x; \mathscr{S})$. **Bounds**: $\overline{w}_1 = 1$, $\overline{w}_t = (2\pi\sigma_v^2)^{-d/2}$.



In [ ]:
# @title Define conditioned random walk function

def run_conditioned_random_walk(T, N, d, sigma_v=0.2, num_trials=1000, seed=0):
  rng = jax.random.PRNGKey(seed)
  norm_const = (2 * jnp.pi * sigma_v**2) ** (-d / 2.0)
  wbar = jnp.concatenate([jnp.ones(1), jnp.full(T - 1, norm_const)])

  def sample_and_run(rng):
    rng_prop, rng_ers = jax.random.split(rng)
    xs = jax.random.uniform(rng_prop, (T, N, d))

    def _w_init(x):
      return jnp.ones(x.shape[0]), 0.0

    def _weight_matrix(x_prev, x_t, t_idx):
      x1sq = jnp.sum(x_prev**2, axis=-1)
      x2sq = jnp.sum(x_t**2, axis=-1)
      dists = x1sq[:, None] + x2sq[None, :] - 2 * x_prev @ x_t.T
      dists = jnp.maximum(dists, 0.0)
      W = jnp.exp(-dists / (2.0 * sigma_v**2)) * norm_const
      return W, 0.0

    def _log_transition(x_next, x_prevs, t_next):
      diff = x_next[None, :] - x_prevs
      return -jnp.sum(diff**2, axis=-1) / (2 * sigma_v**2)

    return ers_engine(rng_ers, xs, _w_init, _weight_matrix, _log_transition, wbar)

  jit_step = jax.jit(sample_and_run)
  return run_experiment(jit_step, rng, num_trials=num_trials)


print('run_conditioned_random_walk defined.')

In [ ]:
# @title Table 1a: Conditioned random walks, d=1

print('Table 1a: d=1')
print(f"{'T':>6}  {'N':>6}  {'p_acc':>8}  {'std':>8}  {'time':>8}")
print('-' * 45)

results_rw_d1 = []
for T in [100, 250, 500, 1000]:
  for N_mult in [1, 2, 5, 10]:
    N = N_mult * T
    p_acc, std_acc, dur = run_conditioned_random_walk(T, N, d=1, num_trials=1000)
    print(f"{T:>6}  {N:>6}  {p_acc:>8.3f}  {std_acc:>8.4f}  {dur:>7.1f}s")
    results_rw_d1.append(dict(d=1, T=T, N=N, N_mult=N_mult, p_acc=p_acc, std=std_acc, time=dur))

In [ ]:
# @title Table 1b: Conditioned random walks, d=2

print('Table 1b: d=2')
print(f"{'T':>6}  {'N':>6}  {'p_acc':>8}  {'std':>8}  {'time':>8}")
print('-' * 45)

results_rw_d2 = []
for T in [100, 250, 500, 1000]:
  for N_mult in [1, 2, 5, 10]:
    N = N_mult * T
    p_acc, std_acc, dur = run_conditioned_random_walk(T, N, d=2, num_trials=1000)
    print(f"{T:>6}  {N:>6}  {p_acc:>8.3f}  {std_acc:>8.4f}  {dur:>7.1f}s")
    results_rw_d2.append(dict(d=2, T=T, N=N, N_mult=N_mult, p_acc=p_acc, std=std_acc, time=dur))


## Example 2: Non-linear time series / Kitagawa model (Section 5.2)

**Model**: $X_t \mid X_{t-1} \sim \mathscr{N}(m_t(X_{t-1}), \sigma_v^2)$ where $m_t(x) = \tfrac{x}{2} + 25\tfrac{x}{1+x^2} + 8\cos(1.2t)$, $Y_t \mid X_t \sim \mathscr{N}(X_t^2/20, \sigma_w^2)$.

**Key**: $f_t$ is **time-inhomogeneous** — `weight_matrix_fn` uses `t_idx`. **Bounds**: $\overline{w}_1 = 1$, $\overline{w}_t = (2\pi\sigma_v^2)^{-1/2}$.



In [ ]:
# @title Kitagawa helpers: densities, proposals, data generation

def log_normal(x, mu, var):
  return -0.5 * (x - mu)**2 / var - 0.5 * jnp.log(2 * jnp.pi * var)


def sample_pi_kappa(rng, kappa, sigma_w2, n_samples, n_max=100):
  sigma_w = jnp.sqrt(sigma_w2)
  lam = kappa / sigma_w

  def sample_positive_lambda(rng):
    rng_n, rng_g, rng_s = jax.random.split(rng, 3)
    ns = jnp.arange(n_max)
    log_weights = (ns * jnp.log(jnp.maximum(jnp.sqrt(2.0) * lam, 1e-30))
                   - jax.lax.lgamma(ns + 1.0)
                   + jax.lax.lgamma((2.0 * ns + 1.0) / 4.0))
    log_weights = log_weights - jax.scipy.special.logsumexp(log_weights)
    weights = jnp.exp(log_weights)
    N_idx = jax.random.choice(rng_n, n_max, shape=(n_samples,), p=weights)
    alpha = (2.0 * N_idx + 1.0) / 4.0
    G = jax.random.gamma(rng_g, alpha)
    signs = (2.0 * jax.random.bernoulli(rng_s, 0.5, (n_samples,)).astype(jnp.float32) - 1.0)
    return signs * (800.0 * sigma_w2 * G) ** 0.25

  def sample_negative_lambda(rng):
    L = -lam
    rng_g, rng_u, rng_s = jax.random.split(rng, 3)
    n_propose = n_samples * 20
    G = jax.random.gamma(rng_g, 0.25, shape=(n_propose,))
    r = (2.0 * G) ** 0.25
    u = jax.random.uniform(rng_u, (n_propose,))
    mask = jnp.log(u) < (-L * r**2)
    signs = (2.0 * jax.random.bernoulli(rng_s, 0.5, (n_propose,)).astype(jnp.float32) - 1.0)
    x_all = signs * jnp.sqrt(20.0 * sigma_w) * r
    sorted_idx = jnp.argsort(jnp.where(mask, 0.0, 1.0))[:n_samples]
    return x_all[sorted_idx]

  return jax.lax.cond(lam >= 0, sample_positive_lambda, sample_negative_lambda, rng)


def generate_kitagawa_data(rng, T, sigma_mu2=10.0, sigma_v2=10.0, sigma_w2=1.0):
  sigma_mu = jnp.sqrt(sigma_mu2)
  sigma_v = jnp.sqrt(sigma_v2)
  sigma_w = jnp.sqrt(sigma_w2)
  times = jnp.arange(1, T + 1, dtype=jnp.float32)
  cos_terms = 8.0 * jnp.cos(1.2 * times)

  def gen_step(carry, t_idx):
    x_prev, rng = carry
    rng, rng_x, rng_y = jax.random.split(rng, 3)
    m = 0.5 * x_prev + 25.0 * x_prev / (1.0 + x_prev**2) + cos_terms[t_idx]
    x = m + sigma_v * jax.random.normal(rng_x)
    y = x**2 / 20.0 + sigma_w * jax.random.normal(rng_y)
    return (x, rng), (x, y)

  rng, rng_init, rng_y0 = jax.random.split(rng, 3)
  x0 = sigma_mu * jax.random.normal(rng_init)
  y0 = x0**2 / 20.0 + sigma_w * jax.random.normal(rng_y0)
  _, (xs_rest, ys_rest) = jax.lax.scan(gen_step, (x0, rng), jnp.arange(1, T))
  return jnp.concatenate([x0[None], xs_rest]), jnp.concatenate([y0[None], ys_rest])


def make_kitagawa_model(xtrue, ytrue, N, sigma_mu2=10.0, sigma_v2=10.0, sigma_w2=1.0):
  T = ytrue.shape[0]
  times = jnp.arange(1, T + 1, dtype=jnp.float32)
  cos_terms = 8.0 * jnp.cos(1.2 * times)
  f_sup = 1.0 / jnp.sqrt(2 * jnp.pi * sigma_v2)
  wbar = jnp.concatenate([jnp.ones(1), jnp.full(T - 1, f_sup)])
  ytilde_1 = ytrue[0] - 10.0 * sigma_w2 / sigma_mu2
  kappa = jnp.concatenate([ytilde_1[None], ytrue[1:]])
  return dict(T=T, N=N, sigma_mu2=sigma_mu2, sigma_v2=sigma_v2, sigma_w2=sigma_w2,
              cos_terms=cos_terms, ytrue=ytrue, xtrue=xtrue, kappa=kappa, wbar=wbar)


def kit_sample_proposals(rng, m):
  T, N = m['T'], m['N']
  def sample_t(rng_t, t_idx):
    return sample_pi_kappa(rng_t, m['kappa'][t_idx], m['sigma_w2'], N)
  return jax.vmap(sample_t)(jax.random.split(rng, T), jnp.arange(T))


print('Kitagawa helpers defined.')

In [ ]:
# @title Example 2: Kitagawa using generic ers_engine

def ers_kitagawa(rng, m):
  T, N = m['T'], m['N']
  sigma_v2 = m['sigma_v2']
  cos_terms = m['cos_terms']
  rng, rng_prop = jax.random.split(rng)
  xs = kit_sample_proposals(rng_prop, m)

  def w_init_fn(x):
    return jnp.ones(N), 0.0

  def weight_matrix_fn(x_prev, x_t, t_idx):
    means = 0.5 * x_prev + 25.0 * x_prev / (1.0 + x_prev**2) + cos_terms[t_idx]
    diffs = x_t[None, :] - means[:, None]
    log_f = -0.5 * diffs**2 / sigma_v2 - 0.5 * jnp.log(2 * jnp.pi * sigma_v2)
    log_f_max = jnp.max(log_f)
    return jnp.exp(log_f - log_f_max), log_f_max

  def log_transition_fn(x_next, x_prevs, t_next):
    means = 0.5 * x_prevs + 25.0 * x_prevs / (1.0 + x_prevs**2) + cos_terms[t_next]
    return -0.5 * (x_next - means)**2 / sigma_v2 - 0.5 * jnp.log(2 * jnp.pi * sigma_v2)

  return ers_engine(rng, xs, w_init_fn, weight_matrix_fn, log_transition_fn, m['wbar'])


print('Kitagawa model (generic engine) defined.')

In [ ]:
# @title Run Kitagawa experiments

FRESH_DATA = True
num_trials = 1000
T_max_kit = 100

rng_kit_data = jax.random.PRNGKey(100)
xtrue_all, ytrue_all = generate_kitagawa_data(rng_kit_data, T=T_max_kit)
print(f'  FRESH_DATA = {FRESH_DATA}')
print(f"\nKitagawa (optimal proposals)")
print(f"{'T':>5}  {'N':>6}  {'p_acc':>8}  {'std':>8}  {'time':>8}")
print('-' * 45)

configs_kit = [
    (25, 25), (25, 125), (25, 250), (25, 500), (25, 1000), (25, 2500),
    (50, 50), (50, 250), (50, 500), (50, 1000), (50, 2000), (50, 5000),
    (100, 100), (100, 500), (100, 1000), (100, 2000), (100, 4000), (100, 10000)]
results_kit = []

@functools.partial(jax.jit, static_argnums=(1, 2))
def fresh_trial(rng, _T, _N):
  rng_data, rng_ers = jax.random.split(rng)
  xt, yt = generate_kitagawa_data(rng_data, T=_T)
  m = make_kitagawa_model(xt, yt, _N)
  accept, _, _ = ers_kitagawa(rng_ers, m)
  return accept

for T, N in configs_kit:
  if not FRESH_DATA:
    m = make_kitagawa_model(xtrue_all[:T], ytrue_all[:T], N)
    def _step(r, _m=m): return ers_kitagawa(r, _m)
    jit_step = jax.jit(_step)
    _ = jit_step(jax.random.PRNGKey(999))
    rng_ers = jax.random.PRNGKey(42)
    accepts = []
    tic = time.time()
    for _ in range(num_trials):
      rng_ers, rng_s = jax.random.split(rng_ers)
      acc, _, _ = jit_step(rng_s)
      accepts.append(float(acc))
    toc = time.time()
    accepts = jnp.array(accepts)
    p_acc = float(jnp.mean(accepts))
    std_acc = float(jnp.std(accepts) / jnp.sqrt(num_trials))
    dur = toc - tic
  else:
    _ = fresh_trial(jax.random.PRNGKey(999), T, N)
    rng = jax.random.PRNGKey(42)
    accepts = []
    tic = time.time()
    for _ in range(num_trials):
      rng, rng_s = jax.random.split(rng)
      acc = fresh_trial(rng_s, T, N)
      accepts.append(float(acc))
    toc = time.time()
    accepts = jnp.array(accepts)
    p_acc = float(jnp.mean(accepts))
    std_acc = float(jnp.std(accepts) / jnp.sqrt(num_trials))
    dur = toc - tic

  print(f"{T:>5}  {N:>6}  {p_acc:>8.3f}  {std_acc:>8.4f}  {dur:>7.1f}s")
  results_kit.append(dict(T=T, N=N, p_acc=p_acc, std=std_acc, time=dur))


## Example 3: Stochastic volatility model (Section 5.3)

**Model**: $\mu(x) = \mathscr{N}(x; 0, \sigma^2/(1-\phi^2))$, $f(x'\mid x) = \mathscr{N}(x'; \phi x, \sigma^2)$, $g(y\mid x) = \mathscr{N}(y; 0, \beta^2\exp(x))$.

Parameters: $\phi=0.95$, $\beta=0.7$, $\sigma=0.3$. **Bounds**: $\overline{w}_1 = \{(1-\phi^2)/(2\pi\sigma^2)\}^{1/2}$, $\overline{w}_t = (2\pi\sigma^2)^{-1/2}$.

  

In [ ]:
# @title SP500 data

ytrue_sv = jnp.array([
    0.9846, 0.1622, 0.1972, -2.2813, -1.3814, 0.2072,
    -2.0451, -1.6635, -3.0438, 1.4388, 3.1379, -0.0311,
    0.8830, -1.7048, 1.2008, 0.1642, 0.4016, -1.2189,
    0.9132, -0.5488, -0.1836, 0.4661, -1.2134, -0.5728,
    0.2962, 0.2609, -0.6297, -1.6304, -0.0514, -2.1855,
    1.1977, -1.0435, -1.3498, 1.6738, 2.8634, 0.0857,
    -1.2161, 0.4134, -0.3813, 0.6336, -2.7096, -1.5558,
    -1.6548, 1.5349, 1.0609, -1.4316, -0.0535, 2.3094,
    2.1805, 0.7270, -0.7654, 0.0768, -0.7804, -1.7760,
    -0.9331, 0.7195, -0.0197, 0.9885, 1.5609, 0.8748,
    -0.9486, -1.8167, 0.5215, 1.9732, 1.8130, -0.5682,
    0.8557, -1.0605, 0.0315, 0.6976, -1.2700, 0.2281,
    -0.2947, 0.4465, 0.5011, -0.0472, -0.4824, 1.8164,
    0.5818, 0.6918, 1.0880, -0.2580, -0.4019, 0.3472,
    -0.7477, 1.1422, -0.2578, -0.7681, -0.2451, 1.2285,
    0.0454, -0.0242, 0.4925, -0.5592, 0.2876, -0.7768,
    0.1309, 0.4553, -1.1482, -1.4005, -0.2831, -1.7473,
    -0.1713, -1.0888, 0.9712, 0.2223, -0.8730, 0.3960,
    0.7747, 3.6642, 1.2905, -0.3528, -0.8341, 0.5771,
    1.3745, 0.3846, -0.0119, -0.0566, 1.4984, 0.8820,
    -0.2562, 1.5303, 0.8348, 1.9202, -0.4338, 0.7907,
    2.5361, -0.8392, 0.9585, -1.3093, 1.3201, 0.0894,
    -1.1572, -0.0466, 0.1861, 0.4393, -1.2191, 1.3497,
    -0.1824, 0.9220, -0.3082, 1.9812, -0.1461, -0.0691,
    -0.2557, -0.5322, -0.7887, 1.2195, -0.2861, 0.0241,
    -0.3969, -1.4945, 0.3621, -0.3649, 0.2452, 0.6375,
    1.7343, -0.2528, -0.0346, -1.0502, 2.1844, -0.1477,
    0.2188, -1.1680, 0.8753, -1.3560, -0.1098, 1.1934,
    0.7308, 0.2075, 1.6728, 0.7274, -0.5110, -1.1027,
    -0.8495, 0.2124, 0.2616, -0.9213, -0.0607, -1.4243,
    0.4486, 1.3102, 0.0605, 0.0736, -0.1893, -0.7288,
    0.3149, 1.2445, -1.9790, 0.2711, -1.3737, -0.8241,
    0.9774, 0.0537, -0.0295, 0.8213, 0.2235, -0.3275,
    0.6725, 1.1719, 0.2223, 1.0835, 0.7389, -0.4551,
    -0.0825, -0.6858, -0.3799, -1.1008, -0.2269, 0.6530,
    -1.1614, 0.2599, 1.2265, -0.5666, -0.4059, -0.9288,
    0.0879, 0.6187, -1.8192, -0.0782, 0.2533, 0.7534,
    -0.8692, 1.8049, -0.1191, -1.1028, 0.2007, 1.0266,
    -0.4854, -0.0984, 0.3268, 0.8663, 0.5612, -0.2225,
    -0.0944, 1.0932, -0.2989, -0.3494, -0.9078, -0.2058,
    0.6108, -0.0079, 0.5811, 0.9197, 0.2892, -0.1781,
    0.0155, -0.5491, 1.4336, -0.0154, -0.3180, -0.5667,
    0.2322, 0.4115, 0.0718, -0.1463, -0.9679, -2.3910,
    0.7832, 2.8988, 0.1893, 0.7231, -0.0812, -0.2008,
    0.9067, -0.0429, -0.2627, -0.8329, -0.5575, -0.2131,
    -0.0103, -0.1363, -1.0374, 0.1377, 0.5826, -0.9729,
    0.5693, -0.0726, 0.3728, 0.1601, 0.0928, -0.5169,
    0.4628, -0.2143, -0.1009, -0.1528, 0.5066, 0.3449,
    -0.2418, -0.9809, -0.8410, -0.4601, 0.3078, -1.0218,
    0.9903, 0.2362, 1.3074, 1.1679, 0.4567, -0.2243,
])

print(f'Loaded {ytrue_sv.shape[0]} SP500 observations')

In [ ]:
# @title Example 3: Stochastic volatility model (Section 5.3)

def run_stochastic_volatility(T, N, ytrue, phi=0.95, beta=0.7, sigma=0.3,
                              num_trials=1000, seed=0):
  rng = jax.random.PRNGKey(seed)
  y = jnp.array(ytrue[:T])
  ss = sigma / jnp.sqrt(1 - phi**2)
  norm_f = (2 * jnp.pi * sigma**2) ** (-0.5)
  norm_mu = (2 * jnp.pi * ss**2) ** (-0.5)
  wbar = jnp.concatenate([jnp.array([norm_mu]), jnp.full(T - 1, norm_f)])

  def sample_and_run(rng):
    rng_prop, rng_ers = jax.random.split(rng)
    y_broad = jnp.broadcast_to(y[:, None], (T, N))
    z = jax.random.normal(rng_prop, (T, N))
    x = jnp.log(y_broad**2) - jnp.log(beta**2) - jnp.log(z**2)
    xs = x[:, :, None]

    def _w_init(x):
      return jnp.exp(-x[:, 0]**2 / (2 * ss**2)) * norm_mu, 0.0

    def _weight_matrix(x_prev, x_t, t_idx):
      x_pred = phi * x_prev
      x1sq = jnp.sum(x_pred**2, axis=-1)
      x2sq = jnp.sum(x_t**2, axis=-1)
      dists = x1sq[:, None] + x2sq[None, :] - 2 * x_pred @ x_t.T
      dists = jnp.maximum(dists, 0.0)
      W = jnp.exp(-dists / (2.0 * sigma**2)) * norm_f
      return W, 0.0

    def _log_transition(x_next, x_prevs, t_next):
      x_pred = phi * x_prevs
      diff = x_next[None, :] - x_pred
      return -jnp.sum(diff**2, axis=-1) / (2 * sigma**2)

    return ers_engine(rng_ers, xs, _w_init, _weight_matrix, _log_transition, wbar)

  jit_step = jax.jit(sample_and_run)
  return run_experiment(jit_step, rng, num_trials=num_trials)


print('\nStochastic volatility model (S&P 500 data)')
print(f"{'T':>5}  {'N':>6}  {'p_acc':>8}  {'std':>8}  {'time':>8}")
print('-' * 45)

configs_sv = [
      (T, N)
      for T in [50, 100, 200, 300]
      for N in [1000, 2000, 5000, 10000]
]

results_sv = []
for T, N in configs_sv:
    p_acc, std_acc, dur = run_stochastic_volatility(T, N, ytrue_sv, num_trials=1000)
    print(f"{T:>5}  {N:>6}  {p_acc:>8.3f}  {std_acc:>8.4f}  {dur:>7.1f}s")
    results_sv.append(dict(T=T, N=N, p_acc=p_acc, std=std_acc, time=dur))